### ЗАДАЧА: Пакетная обработка переводов между кошельками (exceptions + business rules)

Есть набор кошельков и список входящих переводов.
Нужно безопасно обработать пакет операций: валидные переводы применить к балансам,
ошибочные сохранить в отчёт, не останавливая всю обработку.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `TransferError`
   - `TransferFormatError`
   - `AccountNotFoundError`
   - `CurrencyMismatchError`
   - `InsufficientFundsError`
   - `TransferAmountError`.

2. Функцию `parse_transfer(raw)`:
   - формат строки: `transfer_id|from_user|to_user|amount`
   - `amount` должен быть числом и `> 0`
   - при ошибке конвертации использовать `raise ... from ...`.

3. Функцию `apply_transfer(transfer, wallets)`:
   - проверить, что оба пользователя существуют
   - нельзя переводить самому себе
   - валюты кошельков отправителя и получателя должны совпадать
   - у отправителя должно хватать средств
   - при успехе обновить балансы в `wallets`
   - вернуть краткий словарь результата.

4. Функцию `process_batch(rows, wallets)`:
   - для каждой строки вызвать `parse_transfer`, потом `apply_transfer`
   - вернуть `(successes, errors)`
   - ошибки хранить как `(raw, error_type, message)`
   - не прерывать цикл на первой ошибке.

5. Вывести:
   - успешные переводы,
   - ошибки по типам,
   - итоговые балансы,
   - пользователя с максимальным балансом в валюте `USD`.


In [1]:
wallets = {
    'alice': {'currency': 'USD', 'balance': 1200.0},
    'bob': {'currency': 'USD', 'balance': 450.0},
    'carol': {'currency': 'EUR', 'balance': 900.0},
    'dave': {'currency': 'USD', 'balance': 150.0},
}

rows = [
    'TR-100|alice|bob|200',
    'TR-101|bob|dave|700',
    'TR-102|alice|carol|50',
    'TR-103|eve|bob|30',
    'TR-104|dave|dave|10',
    'TR-105|bob|alice|abc',
    'TR-106|bob|dave|100',
]


class TransferError(Exception):
    pass


class TransferFormatError(TransferError):
    pass


class AccountNotFoundError(TransferError):
    pass


class CurrencyMismatchError(TransferError):
    pass


class InsufficientFundsError(TransferError):
    pass


class TransferAmountError(TransferError):
    pass


def parse_transfer(raw):
    # TODO: распарсить строку и вернуть dict перевода
    # TODO: при ошибке конвертации amount использовать raise ... from ...
    parts = raw.split("|")
    if len(parts) != 4:
        raise TransferFormatError("Неверный формат строки")
    transfer_id, from_user, to_user, amount_str = parts
    try:
        amount = float(amount_str)
        if amount <= 0:
            raise TransferAmountError("Сумма должна быть больще 0")
    except ValueError:
        raise TransferAmountError("Некорректное число") from None
    
    return {
        "transfer_id": transfer_id,
        "from_user": from_user,
        "to_user": to_user,
        "amount": amount
    }


def apply_transfer(transfer, wallets):
    from_user = transfer["from_user"]
    to_user = transfer["to_user"]
    amount = transfer["amount"]
    # TODO: проверить существование аккаунтов
    if from_user not in wallets:
        raise AccountNotFoundError(f"Аккаунт {from_user} не найден")
    if to_user not in wallets:
        raise AccountNotFoundError(f"Аккаунт {to_user} не найден")
        
    # TODO: запретить перевод самому себе
    if from_user == to_user:
        raise TransferError("Нельзя переводить самому себе")
    
    sender_wallet = wallets[from_user]
    receiver_wallet = wallets[to_user]
    
    # TODO: проверить совпадение валют
    if sender_wallet["currency"] != receiver_wallet["currency"]:
        raise CurrencyMismatchError("Валюты не совпадают")
    
    
    # TODO: проверить баланс отправителя
    if sender_wallet["balance"] < amount:
        raise InsufficientFundsError("Недостаточно средств")
    
    # TODO: обновить балансы и вернуть dict результата
    sender_wallet["balance"] -= amount
    receiver_wallet["balance"] += amount
    
    return {
        "transfer_id": transfer["transfer_id"],
        "from": from_user,
        "to": to_user,
        "amount": amount,
        "new_balance_from": sender_wallet["balance"],
        "new_balance_to": receiver_wallet["balance"],
         }
    
    


def process_batch(rows, wallets):
    # TODO: вернуть (successes, errors)
    successes = []
    errors = []
    
    for raw in rows:
        try:
            transfer = parse_transfer(raw)
            result = apply_transfer(transfer,wallets)
            successes.append(result)
        except TransferError as e:
            if isinstance(e, TransferFormatError):
                err_type  = "TransferFormatError"
            elif isinstance(e, AccountNotFoundError):
                err_type = "AccountNotFoundError"
            elif isinstance(e, CurrencyMismatchError):
                err_type = "CurrencyMismatchError"
            elif isinstance(e, InsufficientFundsError):
                err_type = "InsufficientFundsError"
            elif isinstance(e, TransferAmountError):
                err_type = "TransferAmountError"
            else:
                err_type = "UnknownError"
            errors.append((raw, err_type, str(e)))
            
    return successes, errors
            


# TODO: вызвать process_batch(rows, wallets)
successes, errors = process_batch(rows, wallets)

# TODO: вывести успешные переводы
print("Успешные переводы:")
for s in successes:
    print(f"{s['transfer_id']}: {s['from']} -> {s['to']} на сумму {s['amount']}. Новое баланс: {s['new_balance_from']}/{s['from']} и {s['new_balance_to']}/{s['to']}.")
print("Ошибки:")

# TODO: вывести ошибки по типам
from collections import Counter
errors_types = Counter(e[1] for e in errors)
for etype, count in errors_types.items():
    print(f"{etype}: {count}")
    
# TODO: вывести итоговые балансы
print("Итоговые балансы:")
for user, wallet in wallets.items():
    print(f"{user}: {wallet['balance']} {wallet['currency']}")

# TODO: найти richest_usd_user
usd_users  = [(u,w["balance"]) for u, w in wallets.items() if w["currency"] == "USD"]
if usd_users:
    richest_user = max(usd_users, key=lambda x: x[1])
    print(f"Пользователь с максимальным балансом USD: {richest_user[0]} с балансом {richest_user[1]}")
else:
    print("Нет пользователей с валютой USD")  

Успешные переводы:
TR-100: alice -> bob на сумму 200.0. Новое баланс: 1000.0/alice и 650.0/bob.
TR-106: bob -> dave на сумму 100.0. Новое баланс: 550.0/bob и 250.0/dave.
Ошибки:
InsufficientFundsError: 1
CurrencyMismatchError: 1
AccountNotFoundError: 1
UnknownError: 1
TransferAmountError: 1
Итоговые балансы:
alice: 1000.0 USD
bob: 550.0 USD
carol: 900.0 EUR
dave: 250.0 USD
Пользователь с максимальным балансом USD: alice с балансом 1000.0
